# Notebook 02 — DistilBERT Sentiment Classification
## NLP Business Case: Automated Customer Reviews
### Ironhack Bootcamp Project

---

In this notebook, I fine-tune **DistilBERT** (`distilbert-base-uncased`) for 3-class sentiment classification
on the preprocessed Amazon Reviews dataset produced by Notebook 01.

**Labels**: 0 = Negative · 1 = Neutral · 2 = Positive

**Model facts**:
- ~66M parameters, 6 Transformer layers
- 40% smaller and 60% faster than BERT-base
- Maintains ~97% of BERT's performance on language benchmarks
- HuggingFace: https://huggingface.co/distilbert/distilbert-base-uncased

---

## Section 0 — Setup & Environment

In [ ]:
# ── 0.1  Detect execution environment ────────────────────────────────────────
# Check whether we are running inside Google Colab or locally.
# This flag controls whether we mount Google Drive and install packages.
try:
    import google.colab          # noqa: F401  (import only for detection)
    IN_COLAB = True              # Running on Colab
except ImportError:
    IN_COLAB = False             # Running locally

print(f"Running in Colab: {IN_COLAB}")

In [ ]:
# ── 0.2  Mount Google Drive (Colab only) ─────────────────────────────────────
# Google Drive is mounted so that all artifacts survive runtime resets.
# The mount point /content/drive is the standard Colab convention.
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")   # Prompts for Google account auth
    print("Google Drive mounted at /content/drive")
else:
    print("Skipping Drive mount — running locally.")

In [ ]:
# ── 0.3  Install required packages ───────────────────────────────────────────
# Install all ML/NLP dependencies.
# The -q flag suppresses verbose output; --upgrade ensures latest versions.
# 'accelerate' is required by the HuggingFace Trainer for multi-GPU / FP16 support.
if IN_COLAB:
    import subprocess
    subprocess.run([
        "pip", "install", "-q", "--upgrade",
        "transformers",    # HuggingFace Transformers (DistilBERT, Trainer, etc.)
        "datasets",        # HuggingFace Datasets (Arrow format loading)
        "torch",           # PyTorch — deep learning backend
        "scikit-learn",    # Evaluation metrics (F1, confusion matrix, etc.)
        "matplotlib",      # Plotting library for visualizations
        "seaborn",         # Statistical data visualization (confusion matrix heatmap)
        "tqdm",            # Progress bars for training and inference loops
        "accelerate",      # HuggingFace Accelerate for Trainer compatibility
    ], check=True)
    print("All packages installed.")
else:
    print("Skipping pip install — manage dependencies locally via requirements.txt.")

In [ ]:
# ── 0.4  Standard library and third-party imports ────────────────────────────
import os                          # OS path utilities
import json                        # JSON I/O for saving metrics
import random                      # Python random seed
import warnings                    # Suppress non-critical warnings
warnings.filterwarnings("ignore")  # Keep output clean during training

import numpy as np                 # Numerical operations
import pandas as pd                # DataFrame for predictions CSV
import matplotlib.pyplot as plt    # Plotting (loss curves, confusion matrix)
import seaborn as sns              # Heatmap-style confusion matrix

import torch                       # PyTorch core — tensor operations and GPU
from torch.utils.data import DataLoader  # Batch loader (used in inference loop)

from datasets import DatasetDict, load_from_disk  # HuggingFace Datasets utilities

from transformers import (
    DistilBertTokenizerFast,            # Fast tokenizer for DistilBERT
    DistilBertForSequenceClassification,# DistilBERT classification head
    TrainingArguments,                  # Configuration for Trainer
    Trainer,                            # High-level training loop abstraction
    DataCollatorWithPadding,            # Dynamic padding for efficient batching
    EarlyStoppingCallback,              # Stop training if eval metric plateaus
)

from sklearn.metrics import (
    accuracy_score,          # Overall fraction of correct predictions
    precision_score,         # Precision per class and weighted
    recall_score,            # Recall per class and weighted
    f1_score,                # F1 per class and weighted
    confusion_matrix,        # N×N matrix of true vs predicted labels
    classification_report,   # Full text report with per-class metrics
)

from tqdm.auto import tqdm  # Auto-selects notebook or terminal progress bar

print("All imports successful.")
print(f"PyTorch version : {torch.__version__}")

In [ ]:
# ── 0.5  Set random seeds for full reproducibility ───────────────────────────
# Fix seeds in Python, NumPy, and PyTorch so results are deterministic across runs.
SEED = 42

random.seed(SEED)                          # Python built-in random module seed
np.random.seed(SEED)                       # NumPy global seed
torch.manual_seed(SEED)                    # PyTorch CPU seed
torch.cuda.manual_seed_all(SEED)           # PyTorch GPU seed (all devices)

# Makes cuDNN deterministic (slight speed trade-off for reproducibility)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False     # Disable auto-tuner for reproducibility

print(f"Random seeds fixed to {SEED}.")

In [ ]:
# ── 0.6  Define all project paths ────────────────────────────────────────────
# All paths are centralised here so changing them once propagates everywhere.
# BASE_DIR  : root of the project on Google Drive (or local equivalent)
# OUTPUT_DIR: Arrow dataset saved by NB01; also receives CSV/JSON outputs
# PLOTS_DIR : PNG files (confusion matrix, loss curve)
# MODELS_DIR: saved fine-tuned model and tokenizer

if IN_COLAB:
    # Google Drive path — adjust the folder name if yours differs
    BASE_DIR = "/content/drive/MyDrive/nlp-project/business-case-01"
else:
    # Local development path — update to your local project root
    BASE_DIR = os.path.expanduser("~/+Dev/nlp-businesscase")

OUTPUT_DIR  = os.path.join(BASE_DIR, "data", "dataset")   # Arrow dataset root
PLOTS_DIR   = os.path.join(BASE_DIR, "plots")             # PNG visualisations
MODELS_DIR  = os.path.join(BASE_DIR, "models")            # Fine-tuned checkpoints

# Create directories if they do not yet exist
os.makedirs(PLOTS_DIR,  exist_ok=True)   # No error if directory already exists
os.makedirs(MODELS_DIR, exist_ok=True)   # Same for models directory

print(f"BASE_DIR   : {BASE_DIR}")
print(f"OUTPUT_DIR : {OUTPUT_DIR}")
print(f"PLOTS_DIR  : {PLOTS_DIR}")
print(f"MODELS_DIR : {MODELS_DIR}")

In [ ]:
# ── 0.7  Detect GPU availability ─────────────────────────────────────────────
# Training on GPU is ~10–20× faster than CPU for Transformer fine-tuning.
# We select the device once here and reuse the variable throughout the notebook.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if DEVICE.type == "cuda":
    gpu_name = torch.cuda.get_device_name(0)  # Human-readable GPU model name
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9  # Bytes → GB
    print(f"GPU detected : {gpu_name}  ({gpu_mem:.1f} GB)")
else:
    print("No GPU found — training will run on CPU (expect slower execution).")

print(f"Active device: {DEVICE}")

---
## Section 1 — Load Dataset

In this section, I load the preprocessed dataset saved by Notebook 01 in HuggingFace Arrow format.
Arrow gives me fast, memory-mapped access to the data — critical when dealing with millions of reviews.
I verify the split sizes and class distribution before moving to tokenization.

In [ ]:
# ── 1.1  Load Arrow dataset from disk ────────────────────────────────────────
# DatasetDict.load_from_disk() reads the Arrow files written by NB01.
# Expected structure: {"train": Dataset, "val": Dataset, "test": Dataset}
dataset = load_from_disk(OUTPUT_DIR)  # Loads all splits at once as a DatasetDict

print("Dataset loaded successfully.")
print(f"Splits found: {list(dataset.keys())}")   # Should print ['train', 'val', 'test']
print()
print(dataset)  # Summary showing columns and row counts per split

In [ ]:
# ── 1.2  Inspect split sizes ─────────────────────────────────────────────────
# Print row counts for each split to confirm NB01's train/val/test proportions.
for split_name, split_data in dataset.items():
    print(f"  {split_name:6s} → {len(split_data):>8,} rows | columns: {split_data.column_names}")

In [ ]:
# ── 1.3  Verify column names ─────────────────────────────────────────────────
# The notebook expects 'text' (review string) and 'label' (0/1/2 integer).
# We assert their presence early to catch schema mismatches before training.
expected_cols = {"text", "label"}  # Minimum required columns

for split_name, split_data in dataset.items():
    actual_cols = set(split_data.column_names)
    missing = expected_cols - actual_cols
    if missing:
        raise ValueError(f"Split '{split_name}' is missing columns: {missing}")

print("All splits contain 'text' and 'label' columns — schema validated.")

In [ ]:
# ── 1.4  Class distribution per split ────────────────────────────────────────
# Understanding class balance is crucial before training.
# Heavy imbalance may require class weights or oversampling.
# Labels: 0 = Negative, 1 = Neutral, 2 = Positive
LABEL_NAMES = {0: "Negative", 1: "Neutral", 2: "Positive"}

print("Class distribution (counts and %):")
print("-" * 50)

for split_name, split_data in dataset.items():
    labels = split_data["label"]               # List of integer labels for this split
    total  = len(labels)                       # Total rows in this split
    counts = pd.Series(labels).value_counts().sort_index()  # Count per class

    print(f"\n  {split_name.upper()}  (n={total:,})")
    for label_id, count in counts.items():
        name = LABEL_NAMES[label_id]
        pct  = 100 * count / total
        bar  = "█" * int(pct / 2)              # ASCII bar proportional to percentage
        print(f"    {label_id} {name:9s}: {count:>7,}  ({pct:5.1f}%)  {bar}")

In [ ]:
# ── 1.5  Plot class distribution bar chart ────────────────────────────────────
# A visual check of balance across splits; saved to PLOTS_DIR for the report.
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=False)
colors = ["#DC2626", "#EA580C", "#0891B2"]   # Red/Orange/Green for Neg/Neu/Pos

for ax, (split_name, split_data) in zip(axes, dataset.items()):
    counts = pd.Series(split_data["label"]).value_counts().sort_index()
    ax.bar(
        [LABEL_NAMES[i] for i in counts.index],  # x-axis: class names
        counts.values,                            # y-axis: counts
        color=colors,
        edgecolor="white",
        linewidth=0.8
    )
    ax.set_title(f"{split_name.capitalize()} Split", fontsize=13, fontweight="bold")
    ax.set_xlabel("Sentiment Class", fontsize=11)
    ax.set_ylabel("Review Count",    fontsize=11)
    ax.tick_params(axis="x", rotation=0)

    # Annotate bars with count values
    for bar in ax.patches:
        ax.text(
            bar.get_x() + bar.get_width() / 2,  # Horizontal centre of bar
            bar.get_height() + 0.01 * max(counts.values),  # Slightly above bar
            f"{int(bar.get_height()):,}",
            ha="center", va="bottom", fontsize=9
        )

fig.suptitle("Class Distribution Across Splits", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()

# Save figure to PLOTS_DIR for inclusion in the final report
plot_path = os.path.join(PLOTS_DIR, "nb02_class_distribution.png")
plt.savefig(plot_path, dpi=150, bbox_inches="tight")  # dpi=150 = print quality
print(f"Plot saved to: {plot_path}")
plt.show()

In [ ]:
# ── 1.6  Compute class weights for imbalance handling ────────────────────────
# If the dataset is imbalanced (Positive >> Neutral or Negative),
# class weights penalise the model less for majority errors,
# improving minority-class recall.
# Formula: weight_i = total / (n_classes * count_i)  [inverse frequency]

train_labels = dataset["train"]["label"]             # Labels from training split
label_counts = pd.Series(train_labels).value_counts().sort_index()  # Counts per class
n_classes    = len(label_counts)                     # Should be 3
n_total      = len(train_labels)                     # Total training samples

class_weights = {
    label_id: n_total / (n_classes * count)
    for label_id, count in label_counts.items()
}

print("Class weights (used to handle imbalance):")
for label_id, weight in class_weights.items():
    print(f"  {label_id} {LABEL_NAMES[label_id]:9s}: {weight:.4f}")

# Convert to tensor for use in custom Trainer later
CLASS_WEIGHTS_TENSOR = torch.tensor(
    [class_weights[i] for i in sorted(class_weights.keys())],  # Ordered [0, 1, 2]
    dtype=torch.float
).to(DEVICE)  # Move to same device as model

print(f"\nClass weights tensor: {CLASS_WEIGHTS_TENSOR}")

In [ ]:
# ── 1.7  Show sample records ──────────────────────────────────────────────────
# Print 3 example rows from the training set to verify text and label quality.
print("Sample records from training split:")
print("=" * 70)

for i in range(3):
    row        = dataset["train"][i]          # Access row by index
    text       = row["text"]                  # Raw review text
    label_id   = row["label"]                 # Integer label (0 / 1 / 2)
    label_name = LABEL_NAMES[label_id]        # Human-readable label

    # Truncate long texts for display purposes only
    display_text = text[:200] + "..." if len(text) > 200 else text

    print(f"\n[Sample {i+1}]")
    print(f"  Label : {label_id} ({label_name})")
    print(f"  Text  : {display_text}")
    print("-" * 70)

---
## Section 2 — Tokenization

In this section, I tokenize all text using `DistilBertTokenizerFast`.
DistilBERT shares BERT's WordPiece vocabulary (30,522 tokens) and uses `[CLS]` and `[SEP]` special tokens.

Key choices:
- `max_length=128`: balances coverage vs. memory footprint (most reviews fit within this)
- `truncation=True`: clips longer texts at 128 tokens
- `padding="max_length"`: pads shorter texts to 128 (uniform batch shapes for the collator)
- `batched=True`: processes multiple examples at once for speed

In [ ]:
# ── 2.1  Load DistilBertTokenizerFast ────────────────────────────────────────
# 'Fast' tokenizer is implemented in Rust via HuggingFace tokenizers library.
# It is ~5× faster than the Python tokenizer for large datasets.
MODEL_NAME = "distilbert-base-uncased"   # Identifier on the HuggingFace Hub

tokenizer = DistilBertTokenizerFast.from_pretrained(
    MODEL_NAME,       # Downloads vocab and config from HuggingFace Hub
)

print(f"Tokenizer loaded: {MODEL_NAME}")
print(f"  Vocabulary size    : {tokenizer.vocab_size:,} tokens")
print(f"  Model max length   : {tokenizer.model_max_length} tokens")
print(f"  CLS token          : '{tokenizer.cls_token}'  (id={tokenizer.cls_token_id})")
print(f"  SEP token          : '{tokenizer.sep_token}'  (id={tokenizer.sep_token_id})")
print(f"  PAD token          : '{tokenizer.pad_token}'  (id={tokenizer.pad_token_id})")

In [ ]:
# ── 2.2  Define tokenization function ────────────────────────────────────────
# This function is applied to every batch via dataset.map().
# max_length=128 keeps GPU memory manageable on Colab (16GB VRAM).
# truncation=True silently clips texts beyond 128 tokens.
# padding="max_length" produces fixed-length tensors — not strictly necessary
#   with DataCollatorWithPadding, but ensures consistency.

MAX_LENGTH = 128   # Token sequence length cap (covers ~95%+ of reviews)

def tokenize_function(batch):
    """
    Tokenise a batch of texts.

    Args:
        batch (dict): A dict with key 'text' mapping to a list of strings.

    Returns:
        dict: Adds 'input_ids', 'attention_mask' (and optionally 'token_type_ids')
              to the batch.  The 'label' column is passed through unchanged.
    """
    return tokenizer(
        batch["text"],             # List of raw review strings in this batch
        max_length=MAX_LENGTH,     # Cap sequence length at 128 tokens
        truncation=True,           # Drop tokens beyond max_length
        padding="max_length",      # Pad shorter sequences to exactly max_length
    )

In [ ]:
# ── 2.3  Apply tokenizer to all splits ───────────────────────────────────────
# .map() applies tokenize_function in batches across the whole DatasetDict.
# batched=True feeds 1000 examples at a time (default) — much faster than row-by-row.
# remove_columns=["text"] drops the raw string column since the model uses token IDs.
# After this cell every split will have: input_ids, attention_mask, label

print("Tokenising dataset — this may take a few minutes...")

tokenized_dataset = dataset.map(
    tokenize_function,          # Function to apply to each batch
    batched=True,               # Process multiple examples per call (faster)
    remove_columns=["text"],    # Drop raw text — model uses input_ids instead
    desc="Tokenizing",          # Label shown in the tqdm progress bar
)

print("\nTokenisation complete.")
print(tokenized_dataset)   # Shows new columns: input_ids, attention_mask, label

In [ ]:
# ── 2.4  Set dataset format to PyTorch tensors ───────────────────────────────
# By default HuggingFace Datasets stores data as Python lists.
# set_format("torch") converts columns to torch.Tensor on-the-fly for the DataLoader.
# columns specifies exactly which columns the Trainer / DataLoader should see.

tokenized_dataset.set_format(
    type="torch",                                     # Return torch.Tensor objects
    columns=["input_ids", "attention_mask", "label"], # Only these columns are returned
)

print("Dataset format set to PyTorch tensors.")
print(f"Train tensor shapes (first sample):")
sample = tokenized_dataset["train"][0]  # Fetch first row as a dict of tensors
for col, val in sample.items():
    print(f"  {col:16s}: shape={tuple(val.shape) if hasattr(val, 'shape') else 'scalar'}, "
          f"dtype={val.dtype}")

In [ ]:
# ── 2.5  Inspect a tokenized example ─────────────────────────────────────────
# Decode the first training token sequence back to words to verify tokenisation.
# This is a sanity check — the decoded text should match the original review.
sample_ids    = tokenized_dataset["train"][0]["input_ids"]   # Tensor of token IDs
decoded_text  = tokenizer.decode(sample_ids, skip_special_tokens=False)  # IDs → words

print("=== Tokenised example ===")
print(f"  Label         : {tokenized_dataset['train'][0]['label'].item()} "
      f"({LABEL_NAMES[tokenized_dataset['train'][0]['label'].item()]})")
print(f"  input_ids     : {sample_ids[:20].tolist()} ... (first 20 of {MAX_LENGTH})")
print(f"  attention_mask: {tokenized_dataset['train'][0]['attention_mask'][:20].tolist()} ...")
print(f"  Decoded tokens: {decoded_text[:300]}...")

---
## Section 3 — Model Setup

In this section, I load `DistilBertForSequenceClassification` from the HuggingFace Hub
and attach a 3-class classification head.
I also define the label ↔ id mappings that let the model produce human-readable predictions.

In [ ]:
# ── 3.1  Define label ↔ id mappings ──────────────────────────────────────────
# HuggingFace models use id2label / label2id for automatic label interpretation
# in pipelines and during evaluation. These are stored in the model config.
id2label = {0: "Negative", 1: "Neutral", 2: "Positive"}   # int → string
label2id = {v: k for k, v in id2label.items()}             # string → int

print("Label mappings:")
print(f"  id2label : {id2label}")
print(f"  label2id : {label2id}")

In [ ]:
# ── 3.2  Load DistilBertForSequenceClassification ────────────────────────────
# Downloads the pre-trained DistilBERT weights (~256 MB) on first run.
# num_labels=3 replaces the default binary head with a 3-class softmax head.
# id2label / label2id are saved to config.json with the model checkpoint.

model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME,        # "distilbert-base-uncased" defined in Section 2
    num_labels=3,      # 3-way classification: Negative / Neutral / Positive
    id2label=id2label, # Embed label names into model config
    label2id=label2id, # Embed reverse mapping into model config
)

# Move model to GPU (if available) for faster training and inference
model = model.to(DEVICE)

print(f"Model loaded and moved to {DEVICE}.")
print(f"Model class : {model.__class__.__name__}")

In [ ]:
# ── 3.3  Count trainable parameters ──────────────────────────────────────────
# Understanding model size helps estimate training time and memory requirements.
# We count both total and trainable parameters separately.

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params    = total_params - trainable_params  # Parameters with requires_grad=False

print(f"Parameter count:")
print(f"  Total parameters     : {total_params:>12,}")
print(f"  Trainable parameters : {trainable_params:>12,}  ({100*trainable_params/total_params:.1f}%)")
print(f"  Frozen parameters    : {frozen_params:>12,}  ({100*frozen_params/total_params:.1f}%)")

In [ ]:
# ── 3.4  Inspect the model architecture ──────────────────────────────────────
# Print a human-readable summary of each sub-module and its parameter count.
# The classification head is a Linear layer on top of the DistilBERT encoder.
print("\nModel architecture overview:")
for name, module in model.named_children():
    # Count parameters for this top-level module
    n_params = sum(p.numel() for p in module.parameters())
    print(f"  {name:20s}  →  {module.__class__.__name__:35s}  params: {n_params:,}")

---
## Section 4 — Training

In this section, I fine-tune DistilBERT using the HuggingFace `Trainer` API.

**Training configuration**:
- Batch size: 32
- Epochs: 3
- Learning rate: 2e-5 (standard for Transformer fine-tuning)
- Weight decay: 0.01 (L2 regularisation to prevent overfitting)
- Warmup: 10% of total steps (linearly increases LR from 0 to 2e-5)
- Evaluation: after each epoch (on validation split)
- Best model: selected by lowest `eval_loss`
- Class weights: applied via a custom Trainer subclass to handle label imbalance

In [ ]:
# ── 4.1  Define compute_metrics callback ─────────────────────────────────────
# The Trainer calls this function at the end of each evaluation step.
# It receives a named tuple with predictions (logits) and ground-truth labels.
# We return a dict of metrics that are logged to the training history.

def compute_metrics(eval_pred):
    """
    Compute accuracy and weighted F1 from model logits.

    Args:
        eval_pred: EvalPrediction namedtuple with fields:
            - predictions: np.ndarray of shape (n, num_labels) — raw logits
            - label_ids  : np.ndarray of shape (n,) — ground-truth integers

    Returns:
        dict: {"accuracy": float, "f1": float}
    """
    logits, labels = eval_pred           # Unpack the EvalPrediction namedtuple
    predictions = np.argmax(logits, axis=-1)  # Convert logits → predicted class IDs

    accuracy = accuracy_score(labels, predictions)  # Fraction of correct predictions

    f1 = f1_score(
        labels, predictions,
        average="weighted",   # Weighted by class support — handles imbalance fairly
        zero_division=0,      # Return 0 instead of error for classes with no predictions
    )

    return {"accuracy": accuracy, "f1": f1}  # Trainer logs these after every eval

In [ ]:
# ── 4.2  Custom Trainer with class-weighted loss ─────────────────────────────
# We subclass Trainer and override compute_loss to apply class weights.
# This makes the model pay more attention to under-represented classes
# (typically Neutral reviews) during back-propagation.

import torch.nn as nn  # Neural network building blocks (CrossEntropyLoss)

class WeightedTrainer(Trainer):
    """
    HuggingFace Trainer subclass that applies class weights to the cross-entropy loss.

    The parent Trainer uses a plain CrossEntropyLoss (equal weight per class).
    Here we pass CLASS_WEIGHTS_TENSOR to CrossEntropyLoss so that minority-class
    errors contribute more to the gradient.
    """

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        """
        Override the loss computation to use weighted cross-entropy.

        Args:
            model      : The DistilBERT model being trained.
            inputs     : Dict of tensors (input_ids, attention_mask, labels).
            return_outputs: Whether to also return model outputs (logits).

        Returns:
            loss scalar, or (loss, outputs) if return_outputs=True.
        """
        labels = inputs.pop("labels")      # Extract ground-truth labels from batch

        outputs = model(**inputs)          # Forward pass — returns logits (and hidden states)
        logits  = outputs.logits           # Shape: (batch_size, num_labels)

        # Weighted cross-entropy loss
        loss_fn = nn.CrossEntropyLoss(
            weight=CLASS_WEIGHTS_TENSOR    # Per-class weight tensor (device-aware)
        )
        loss = loss_fn(logits, labels)     # Scalar loss for back-propagation

        return (loss, outputs) if return_outputs else loss

In [ ]:
# ── 4.3  Compute training steps and warmup ────────────────────────────────────
# TrainingArguments needs the number of warmup steps explicitly.
# Warmup avoids large gradient steps early in fine-tuning by ramping the LR up.

BATCH_SIZE    = 32      # Samples processed per gradient update
NUM_EPOCHS    = 3       # Full passes over the training set
LEARNING_RATE = 2e-5    # Peak learning rate after warmup
WEIGHT_DECAY  = 0.01    # L2 regularisation coefficient

# Total gradient updates = (train_size / batch_size) * num_epochs
n_train_samples = len(tokenized_dataset["train"])  # Training set size
steps_per_epoch = n_train_samples // BATCH_SIZE     # Steps in one epoch (floor div)
total_steps     = steps_per_epoch * NUM_EPOCHS      # Total training steps

# Use 10% of total steps for LR warmup (a common heuristic for fine-tuning)
warmup_steps = int(0.10 * total_steps)

print(f"Training configuration:")
print(f"  Training samples    : {n_train_samples:,}")
print(f"  Batch size          : {BATCH_SIZE}")
print(f"  Epochs              : {NUM_EPOCHS}")
print(f"  Steps per epoch     : {steps_per_epoch:,}")
print(f"  Total steps         : {total_steps:,}")
print(f"  Warmup steps (10%)  : {warmup_steps:,}")
print(f"  Learning rate       : {LEARNING_RATE}")
print(f"  Weight decay        : {WEIGHT_DECAY}")

In [ ]:
# ── 4.4  Define TrainingArguments ────────────────────────────────────────────
# TrainingArguments centralises all hyperparameters and logging/saving policies.
# The Trainer reads from this object — no manual training loop required.

CHECKPOINT_DIR = os.path.join(MODELS_DIR, "distilbert_checkpoints")  # Interim saves

training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,               # Where to write checkpoints and logs

    # Batch sizes
    per_device_train_batch_size=BATCH_SIZE,  # Train batch per GPU/CPU device
    per_device_eval_batch_size=BATCH_SIZE,   # Eval batch (no gradient, can be larger)

    # Epochs and learning rate schedule
    num_train_epochs=NUM_EPOCHS,             # Number of full training passes
    learning_rate=LEARNING_RATE,             # Peak LR after warmup
    weight_decay=WEIGHT_DECAY,               # L2 regularisation strength
    warmup_steps=warmup_steps,               # Linear LR warmup duration

    # Evaluation and checkpointing
    evaluation_strategy="epoch",             # Evaluate on val set after every epoch
    save_strategy="epoch",                   # Save checkpoint after every epoch
    load_best_model_at_end=True,             # Restore best checkpoint when done
    metric_for_best_model="eval_loss",       # Use validation loss to select best model
    greater_is_better=False,                 # Lower eval_loss = better model

    # Logging
    logging_dir=os.path.join(CHECKPOINT_DIR, "logs"),  # TensorBoard log directory
    logging_steps=50,                        # Log metrics every 50 steps
    report_to="none",                        # Disable W&B / MLflow — log locally only

    # Reproducibility
    seed=SEED,                               # Forward seed to Trainer

    # Mixed precision (speeds up training on GPU with negligible accuracy loss)
    fp16=torch.cuda.is_available(),          # Enable FP16 only if GPU is available

    # Save disk space: keep only the 2 best checkpoints
    save_total_limit=2,                      # Overwrite older checkpoints
)

print("TrainingArguments defined.")
print(f"  Checkpoints saved to: {CHECKPOINT_DIR}")

In [ ]:
# ── 4.5  Instantiate DataCollatorWithPadding ──────────────────────────────────
# DataCollatorWithPadding dynamically pads each batch to the longest sequence
# in that batch (rather than a global max_length).
# This reduces unnecessary computation on short-sequence batches.

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,      # Needs the tokenizer to know the PAD token ID
    return_tensors="pt",      # Return PyTorch tensors (not numpy or tf)
)

print("DataCollatorWithPadding instantiated.")

In [ ]:
# ── 4.6  Instantiate the WeightedTrainer ─────────────────────────────────────
# Trainer wraps the training loop, gradient accumulation, checkpointing,
# evaluation, and logging — we only need to provide the components.

trainer = WeightedTrainer(
    model=model,                                  # DistilBERT with classification head
    args=training_args,                           # All hyperparameters and paths
    train_dataset=tokenized_dataset["train"],     # Tokenised training split
    eval_dataset=tokenized_dataset["val"],        # Tokenised validation split
    tokenizer=tokenizer,                          # For DataCollator and saving
    data_collator=data_collator,                  # Dynamic padding per batch
    compute_metrics=compute_metrics,              # Accuracy + weighted F1 per epoch
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=2)  # Stop if no val improvement
        # Patience=2: if 2 consecutive epochs show no eval_loss improvement, halt
    ],
)

print("WeightedTrainer instantiated — ready to train.")

In [ ]:
# ── 4.7  Train the model ──────────────────────────────────────────────────────
# trainer.train() runs the full fine-tuning loop.
# Progress is shown via tqdm bars (one per epoch).
# After each epoch, eval metrics are printed automatically.
# Expected duration on a T4 GPU (Colab): ~10–20 min for 3 epochs.

print("Starting fine-tuning...")
print("=" * 60)

train_result = trainer.train()  # Returns a TrainOutput namedtuple

print("=" * 60)
print("Training complete!")
print(f"  Total runtime       : {train_result.metrics['train_runtime']:.1f} seconds")
print(f"  Samples per second  : {train_result.metrics['train_samples_per_second']:.1f}")
print(f"  Final training loss : {train_result.metrics['train_loss']:.4f}")

In [ ]:
# ── 4.8  Extract and store training history ───────────────────────────────────
# trainer.state.log_history contains dicts logged at each step and epoch.
# We separate train-step logs (have 'loss') from eval-epoch logs (have 'eval_loss').

log_history = trainer.state.log_history  # List of dicts

# Training step logs: logged every `logging_steps` steps
train_logs = [
    entry for entry in log_history
    if "loss" in entry and "eval_loss" not in entry  # Step-level training loss
]

# Evaluation epoch logs: logged once per epoch after evaluation
eval_logs = [
    entry for entry in log_history
    if "eval_loss" in entry  # Epoch-level validation metrics
]

print(f"Training log entries   : {len(train_logs)}")
print(f"Evaluation log entries : {len(eval_logs)}")
print()
print("Epoch-level evaluation summary:")
print(f"{'Epoch':>6} | {'eval_loss':>10} | {'accuracy':>9} | {'f1':>8}")
print("-" * 45)
for entry in eval_logs:
    print(f"  {entry.get('epoch', '?'):>4.0f} | "
          f"{entry.get('eval_loss', float('nan')):>10.4f} | "
          f"{entry.get('eval_accuracy', float('nan')):>9.4f} | "
          f"{entry.get('eval_f1', float('nan')):>8.4f}")

---
## Section 5 — Evaluation

In this section, I evaluate the best checkpoint on the held-out **test set**.
I compute per-class and weighted-average metrics, generate a confusion matrix heatmap,
and save all results to a JSON file for the final report.

In [ ]:
# ── 5.1  Run Trainer evaluation on test set ──────────────────────────────────
# trainer.evaluate() runs a forward pass over the full split and returns
# the metric dict from compute_metrics plus the loss.

print("Evaluating best checkpoint on test set...")
test_metrics = trainer.evaluate(eval_dataset=tokenized_dataset["test"])

print("\nTest set metrics (from Trainer):")
for k, v in test_metrics.items():
    print(f"  {k:30s}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

In [ ]:
# ── 5.2  Generate predictions on test set ────────────────────────────────────
# trainer.predict() returns logits for the entire test set in one call.
# We convert logits → class predictions with argmax.

print("Generating predictions on test set...")
pred_output = trainer.predict(tokenized_dataset["test"])  # Returns PredictionOutput

test_logits = pred_output.predictions    # Raw logits: shape (n_test, 3)
test_labels = pred_output.label_ids      # Ground-truth integer labels: shape (n_test,)
test_preds  = np.argmax(test_logits, axis=-1)   # Predicted class IDs: shape (n_test,)
test_probs  = torch.softmax(
    torch.tensor(test_logits), dim=-1
).numpy()  # Convert logits → probabilities: shape (n_test, 3)

print(f"Predictions generated for {len(test_preds):,} test samples.")

In [ ]:
# ── 5.3  Compute detailed classification metrics ──────────────────────────────
# sklearn provides per-class and macro/weighted averages.
# 'weighted' weights each class by its support (number of true instances) —
# appropriate for imbalanced datasets.

class_names = [id2label[i] for i in range(3)]  # ["Negative", "Neutral", "Positive"]

accuracy  = accuracy_score(test_labels, test_preds)  # Overall fraction correct

precision_per_class = precision_score(test_labels, test_preds, average=None,          zero_division=0)
recall_per_class    = recall_score(   test_labels, test_preds, average=None,          zero_division=0)
f1_per_class        = f1_score(       test_labels, test_preds, average=None,          zero_division=0)

precision_weighted  = precision_score(test_labels, test_preds, average="weighted",    zero_division=0)
recall_weighted     = recall_score(   test_labels, test_preds, average="weighted",    zero_division=0)
f1_weighted         = f1_score(       test_labels, test_preds, average="weighted",    zero_division=0)

print("Classification metrics on test set:")
print(f"\n  Overall Accuracy : {accuracy:.4f}  ({100*accuracy:.2f}%)")
print(f"\n  {'Class':10s} | {'Precision':>10} | {'Recall':>8} | {'F1-score':>9}")
print("  " + "-" * 47)
for i, name in enumerate(class_names):
    print(f"  {name:10s} | {precision_per_class[i]:>10.4f} | "
          f"{recall_per_class[i]:>8.4f} | {f1_per_class[i]:>9.4f}")
print("  " + "-" * 47)
print(f"  {'Weighted':10s} | {precision_weighted:>10.4f} | "
      f"{recall_weighted:>8.4f} | {f1_weighted:>9.4f}")

In [ ]:
# ── 5.4  Print full sklearn classification report ─────────────────────────────
# classification_report provides a formatted table with all metrics per class.
# It is the most complete single-function summary for a classification task.

report = classification_report(
    test_labels,
    test_preds,
    target_names=class_names,  # Use human-readable names in the output table
    digits=4,                  # Show 4 decimal places for precision
)

print("Full Classification Report (test set):")
print("=" * 60)
print(report)

In [ ]:
# ── 5.5  Plot and save confusion matrix ──────────────────────────────────────
# The confusion matrix shows how predictions distribute across true classes.
# Off-diagonal cells reveal where the model confuses classes (e.g., Neutral ↔ Positive).
# Normalised by true class (rows) so each row sums to 1.0.

cm      = confusion_matrix(test_labels, test_preds)  # Raw counts: shape (3, 3)
cm_norm = cm.astype("float") / cm.sum(axis=1, keepdims=True)  # Row-normalise → rates

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, data, title, fmt in zip(
    axes,
    [cm, cm_norm],
    ["Confusion Matrix (Counts)", "Confusion Matrix (Normalised)"],
    ["d", ".2f"],  # Integer format for counts, 2-decimal float for normalised
):
    sns.heatmap(
        data,
        annot=True,          # Show values inside each cell
        fmt=fmt,             # Number format
        cmap="Blues",        # Sequential blue palette
        xticklabels=class_names,   # Column labels = predicted classes
        yticklabels=class_names,   # Row labels = true classes
        linewidths=0.5,      # Thin grid lines between cells
        ax=ax,
    )
    ax.set_title(title, fontsize=13, fontweight="bold", pad=12)
    ax.set_xlabel("Predicted Label", fontsize=11)
    ax.set_ylabel("True Label",      fontsize=11)

fig.suptitle("DistilBERT Sentiment Classification — Test Set", fontsize=14, fontweight="bold")
plt.tight_layout()

# Save confusion matrix figure
cm_path = os.path.join(PLOTS_DIR, "nb02_confusion_matrix.png")
plt.savefig(cm_path, dpi=150, bbox_inches="tight")  # High-res PNG for the report
print(f"Confusion matrix saved to: {cm_path}")
plt.show()

In [ ]:
# ── 5.6  Save all test metrics to JSON ───────────────────────────────────────
# Persisting metrics as JSON enables comparison with NB03 (RoBERTa) later.
# The JSON is also imported by the master report notebook.

metrics_dict = {
    "model"            : MODEL_NAME,
    "test_accuracy"    : float(accuracy),           # Overall accuracy
    "precision_weighted": float(precision_weighted), # Weighted precision
    "recall_weighted"  : float(recall_weighted),    # Weighted recall
    "f1_weighted"      : float(f1_weighted),        # Weighted F1
    "per_class": {
        name: {
            "precision": float(precision_per_class[i]),
            "recall"   : float(recall_per_class[i]),
            "f1"       : float(f1_per_class[i]),
        }
        for i, name in enumerate(class_names)
    },
    "classification_report": report,  # Full text of sklearn report
}

metrics_path = os.path.join(OUTPUT_DIR, "metrics_distilbert.json")  # Persistent storage

with open(metrics_path, "w") as f:
    json.dump(metrics_dict, f, indent=2)  # indent=2 for human-readable JSON

print(f"Metrics saved to: {metrics_path}")

---
## Section 6 — Inference & Save Predictions

In this section, I run full inference over the test set and export the results to a CSV file.
The `predictions_distilbert.csv` feeds into Notebook 05 (summarisation) for downstream analysis.

I also save the fine-tuned model and tokenizer to `MODELS_DIR/distilbert_sentiment/`
so they can be reloaded without re-training.

In [ ]:
# ── 6.1  Retrieve raw test texts for the output CSV ──────────────────────────
# The tokenized_dataset dropped the 'text' column (see Section 2).
# We retrieve original texts from the un-tokenized dataset['test'] split.
test_texts = dataset["test"]["text"]   # List of original review strings

print(f"Test texts retrieved: {len(test_texts):,} examples")
print(f"Sample text (first): {test_texts[0][:100]}...")

In [ ]:
# ── 6.2  Build predictions DataFrame ─────────────────────────────────────────
# We combine original texts, true labels, predicted labels, and confidence scores.
# 'confidence' is the softmax probability of the predicted class — useful for
# downstream filtering (e.g., keep only high-confidence predictions).

# Confidence = max probability across the 3 classes for each example
confidence_scores = test_probs.max(axis=1)  # Shape: (n_test,)  values in [0, 1]

predictions_df = pd.DataFrame({
    "text"           : test_texts,                                # Original review text
    "true_label"     : [id2label[i] for i in test_labels],       # "Negative/Neutral/Positive"
    "predicted_label": [id2label[i] for i in test_preds],        # Same format for predictions
    "confidence"     : confidence_scores.round(4),               # 4 decimal precision
})

print(f"Predictions DataFrame shape: {predictions_df.shape}")
print(f"\nColumn dtypes:")
print(predictions_df.dtypes)
print(f"\nFirst 3 rows:")
print(predictions_df.head(3).to_string(index=False))

In [ ]:
# ── 6.3  Save predictions CSV to OUTPUT_DIR ───────────────────────────────────
# This CSV is the primary output consumed by NB05 (review summarisation).
# index=False omits the DataFrame row index from the file.

csv_path = os.path.join(OUTPUT_DIR, "predictions_distilbert.csv")

predictions_df.to_csv(
    csv_path,
    index=False,     # Do not write the integer row index as a column
    encoding="utf-8" # UTF-8 to preserve special characters in review text
)

print(f"Predictions saved to: {csv_path}")
print(f"  Rows    : {len(predictions_df):,}")
print(f"  Columns : {list(predictions_df.columns)}")
print(f"  File size: {os.path.getsize(csv_path) / 1e6:.2f} MB")

In [ ]:
# ── 6.4  Save fine-tuned model and tokenizer ─────────────────────────────────
# Saving both model weights and tokenizer to the same directory allows reloading
# with a single from_pretrained() call pointing at that directory.

MODEL_SAVE_DIR = os.path.join(MODELS_DIR, "distilbert_sentiment")

model.save_pretrained(MODEL_SAVE_DIR)          # Saves config.json + pytorch_model.bin
tokenizer.save_pretrained(MODEL_SAVE_DIR)      # Saves vocab.txt + tokenizer_config.json

print(f"Fine-tuned model saved to: {MODEL_SAVE_DIR}")
print("\nFiles in model directory:")
for fname in sorted(os.listdir(MODEL_SAVE_DIR)):
    fsize = os.path.getsize(os.path.join(MODEL_SAVE_DIR, fname))
    print(f"  {fname:40s}  {fsize / 1e6:8.2f} MB")

In [ ]:
# ── 6.5  Verify model reload (smoke test) ────────────────────────────────────
# Load the saved model and tokenizer from disk and run one prediction
# to confirm the checkpoint is valid and produces sensible output.

from transformers import pipeline  # High-level inference pipeline helper

# Build inference pipeline from the saved directory (no re-download needed)
sentiment_pipeline = pipeline(
    "text-classification",
    model=MODEL_SAVE_DIR,            # Load our fine-tuned weights
    tokenizer=MODEL_SAVE_DIR,        # Load our saved tokenizer
    device=0 if DEVICE.type == "cuda" else -1,  # 0 = GPU, -1 = CPU
)

# Test with one positive and one negative review
test_reviews = [
    "This product is absolutely fantastic! Best purchase I have ever made.",
    "Terrible quality, broke after one day. Complete waste of money.",
]

results = sentiment_pipeline(test_reviews, truncation=True, max_length=MAX_LENGTH)

print("Smoke test — model reload and inference:")
print("-" * 60)
for review, result in zip(test_reviews, results):
    print(f"  Text  : {review[:60]}...")
    print(f"  Pred  : {result['label']}  (score={result['score']:.4f})")
    print()
print("Model reload verified successfully.")

---
## Section 7 — Results Summary

In this final section, I consolidate all results into a readable summary:
- Final metrics table
- 5 correct predictions and 5 incorrect predictions
- Training loss curve plot

In [ ]:
# ── 7.1  Print final metrics summary table ────────────────────────────────────
# A concise table suitable for the final project report.

print("╔══════════════════════════════════════════════════════════╗")
print("║        DistilBERT Sentiment Classification — Results     ║")
print("╠══════════════════════════════════════════════════════════╣")
print(f"║  Model           : {MODEL_NAME:<38s}║")
print(f"║  Test Accuracy   : {accuracy*100:>6.2f}%                                ║")
print(f"║  Precision (W)   : {precision_weighted*100:>6.2f}%                                ║")
print(f"║  Recall (W)      : {recall_weighted*100:>6.2f}%                                ║")
print(f"║  F1-score (W)    : {f1_weighted*100:>6.2f}%                                ║")
print("╠══════════════════════════════════════════════════════════╣")
print(f"║  {'Class':10s}   {'Precision':>10}   {'Recall':>8}   {'F1':>8}      ║")
print("╠══════════════════════════════════════════════════════════╣")
for i, name in enumerate(class_names):
    print(f"║  {name:10s}   {precision_per_class[i]*100:>9.2f}%   "
          f"{recall_per_class[i]*100:>7.2f}%   {f1_per_class[i]*100:>7.2f}%      ║")
print("╚══════════════════════════════════════════════════════════╝")

In [ ]:
# ── 7.2  Show 5 correct and 5 incorrect predictions ─────────────────────────
# Examining errors and successes gives intuition about model strengths/weaknesses.

# Add a boolean 'correct' column for easy filtering
predictions_df["correct"] = (
    predictions_df["true_label"] == predictions_df["predicted_label"]
)

correct_samples   = predictions_df[predictions_df["correct"]  == True ].head(5)
incorrect_samples = predictions_df[predictions_df["correct"]  == False].head(5)

def print_examples(title, df):
    """Print a formatted table of example predictions."""
    print(f"\n{'='*70}")
    print(f"  {title}")
    print(f"{'='*70}")
    for _, row in df.iterrows():
        # Truncate text for display — full text is in the CSV
        display = row["text"][:120] + "..." if len(row["text"]) > 120 else row["text"]
        print(f"  Text      : {display}")
        print(f"  True      : {row['true_label']:10s}  |  Predicted: {row['predicted_label']:10s}  |  Confidence: {row['confidence']:.4f}")
        print(f"  {'-'*68}")

print_examples("CORRECT Predictions (5 examples)", correct_samples)
print_examples("INCORRECT Predictions (5 examples)", incorrect_samples)

In [ ]:
# ── 7.3  Plot training loss curve ─────────────────────────────────────────────
# The loss curve shows whether the model converged and whether it over-fitted.
# A healthy curve shows monotonically decreasing train loss and stable val loss.

# Extract step-level training loss
train_steps  = [entry["step"] for entry in train_logs]   # Step numbers
train_losses = [entry["loss"] for entry in train_logs]   # Loss values

# Extract epoch-level evaluation loss (one point per epoch)
eval_epochs  = [entry["epoch"]     for entry in eval_logs]  # Epoch floats
eval_losses  = [entry["eval_loss"] for entry in eval_logs]  # Validation loss

fig, ax = plt.subplots(figsize=(10, 5))

# Plot training loss (step-level — dense)
ax.plot(
    train_steps, train_losses,
    color="#0891B2", linewidth=1.5,
    label="Training Loss (per step)",
    alpha=0.85
)

# Plot validation loss (epoch-level — sparse) as larger markers
if eval_losses:
    # Map epoch numbers to approximate step numbers for x-axis alignment
    eval_steps_approx = [
        int(ep * steps_per_epoch) for ep in eval_epochs
    ]
    ax.plot(
        eval_steps_approx, eval_losses,
        color="#334155", linewidth=2.0, marker="o", markersize=8,
        label="Validation Loss (per epoch)",
    )

ax.set_title("DistilBERT Fine-tuning — Loss Curves", fontsize=14, fontweight="bold")
ax.set_xlabel("Training Step",  fontsize=12)
ax.set_ylabel("Loss",           fontsize=12)
ax.legend(fontsize=11)
ax.grid(True, linestyle="--", alpha=0.5)  # Subtle grid for readability

plt.tight_layout()

loss_plot_path = os.path.join(PLOTS_DIR, "nb02_training_loss_curve.png")
plt.savefig(loss_plot_path, dpi=150, bbox_inches="tight")
print(f"Training loss curve saved to: {loss_plot_path}")
plt.show()

In [ ]:
# ── 7.4  Final output file checklist ─────────────────────────────────────────
# Verify that all expected output files were created before wrapping up.

expected_outputs = {
    "Predictions CSV"        : os.path.join(OUTPUT_DIR, "predictions_distilbert.csv"),
    "Metrics JSON"           : os.path.join(OUTPUT_DIR, "metrics_distilbert.json"),
    "Confusion Matrix PNG"   : os.path.join(PLOTS_DIR,  "nb02_confusion_matrix.png"),
    "Loss Curve PNG"         : os.path.join(PLOTS_DIR,  "nb02_training_loss_curve.png"),
    "Class Distribution PNG" : os.path.join(PLOTS_DIR,  "nb02_class_distribution.png"),
    "Saved Model Dir"        : os.path.join(MODELS_DIR, "distilbert_sentiment"),
}

print("Output file checklist:")
print("-" * 55)
all_ok = True
for label, path in expected_outputs.items():
    exists = os.path.exists(path)
    status = "✓  OK" if exists else "✗  MISSING"
    if not exists:
        all_ok = False
    print(f"  {label:30s}  {status}")

print("-" * 55)
if all_ok:
    print("All output files present — Notebook 02 complete!")
else:
    print("WARNING: Some output files are missing. Check the cells above for errors.")

---
## Notebook 02 Complete ✓

**Summary of work done in this notebook:**

- Loaded the Arrow-format dataset produced by Notebook 01 (train / val / test splits)
- Tokenised all review texts with `DistilBertTokenizerFast` (max_length=128)
- Fine-tuned `DistilBertForSequenceClassification` (3-class) using the HuggingFace `Trainer` API
- Applied class-weighted cross-entropy loss to handle label imbalance
- Evaluated on the held-out test set (Accuracy, Precision, Recall, F1 per class + weighted)
- Generated a confusion matrix heatmap and training loss curve (saved as PNG)
- Saved predictions to `predictions_distilbert.csv` (feeds into NB05 summarisation)
- Persisted the fine-tuned model + tokenizer to `models/distilbert_sentiment/`

**Next step:** Notebook 03 — RoBERTa Sentiment Classification (higher-capacity comparison model)